# Aive orchestration decisions

One tiny encoder answers every judgement question the orchestration utilities ask
(`OrchestrationQuestion` in `OrchestrationDecisions.kt`), each through its own linear head, in one
forward pass per text. The rest of orchestration is exact code.

The notebook trains each candidate encoder, gates it, exports it to INT8 ONNX, checks the INT8 model
against the float one, and releases the **smallest** candidate that passes as
`orchestration-decisions-v1` (`model.onnx`, `tokenizer.json`, `config.json`). Runs on Colab (work kept
in Google Drive) or Kaggle, GPU or CPU; a T4 trains every candidate in well under an hour.

Generated by `tools/decision_training/make_notebook.py`; edit that file, not this notebook.

In [ ]:
%pip install -q "transformers>=4.48,<4.58" onnx "onnxruntime>=1.20" "onnxscript"

In [ ]:
import json, math, os, random, shutil, subprocess, sys, tarfile, time, hashlib
from pathlib import Path

REPOSITORY = "HereLiesAz/aive"
REF = "main"
RELEASE_TAG = "orchestration-decisions-v1"
UPLOAD = "auto"                      # "auto": publish when a GITHUB_TOKEN secret is set
# Smallest first: the first candidate that passes every gate is released, so order is preference.
CANDIDATES = [
    "google/bert_uncased_L-2_H-128_A-2",   # 4.4M parameters
    "google/bert_uncased_L-4_H-256_A-4",   # 11M
    "jhu-clsp/ettin-encoder-17m",          # 17M, ModernBERT
    "sentence-transformers/all-MiniLM-L6-v2",  # 22M
    "jhu-clsp/ettin-encoder-32m",          # 32M
]
TRAIN_ALL = False                    # True: train every candidate for the comparison table, not only up to the first pass
ROWS_PER_CLASS = 600
MAX_LENGTH = 96                      # tokens; the longest criteria lists fit
EPOCHS = 12
PATIENCE = 3                         # epochs without a better worst-class validation score
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
MIN_CONFIDENCE = 0.8                 # DecisionInformedOrchestrationUtilities.DEFAULT_MIN_CONFIDENCE
# Gates, per question. The runtime acts only on answers at or above MIN_CONFIDENCE, so what matters is
# how often those are right; unconfident answers fall back to the heuristic.
GATES = {
    "min_class_accuracy": 0.85,      # test split, every class of every question
    "min_confident_accuracy": 0.97,  # test split, answers at or above MIN_CONFIDENCE
    "min_adversarial_confident_accuracy": 0.8,
    "max_int8_accuracy_drop": 0.02,  # INT8 ONNX against the float model, test split
}

def work_root():
    try:
        from google.colab import drive
    except ImportError:
        return Path("/kaggle/working") if Path("/kaggle").exists() else Path.cwd()
    drive.mount("/content/drive")
    return Path("/content/drive/MyDrive")

WORK = work_root() / "aive-decisions"
WORK.mkdir(parents=True, exist_ok=True)

source = WORK / "aive-source"
if not (source / "tools/decision_training/generate_corpus.py").exists():
    shutil.rmtree(source, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF, f"https://github.com/{REPOSITORY}", str(source)], check=True)
CORPUS = WORK / "corpus"
subprocess.run([sys.executable, str(source / "tools/decision_training/generate_corpus.py"), str(CORPUS),
                "--rows-per-class", str(ROWS_PER_CLASS)], check=True, stdout=subprocess.DEVNULL)
MANIFEST = json.loads((CORPUS / "manifest.json").read_text())
QUESTIONS = MANIFEST["questions"]                 # id -> options, in OrchestrationQuestion order
ROWS = [json.loads(line) for line in (CORPUS / "decisions.jsonl").open()]
SPLITS = {name: [r for r in ROWS if r["split"] == name] for name in ("train", "validation", "test", "adversarial")}
print({name: len(rows) for name, rows in SPLITS.items()})

In [ ]:
import torch
from torch import nn
from transformers import AutoModel, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
QUESTION_IDS = list(QUESTIONS)
OFFSETS, offset = {}, 0
for q in QUESTION_IDS:
    OFFSETS[q] = (offset, offset + len(QUESTIONS[q]))
    offset += len(QUESTIONS[q])
TOTAL_OPTIONS = offset

class DecisionModel(nn.Module):
    """Encoder, mean pooling, one linear head per question; logits of every head side by side,
    divided by each question's temperature, so a softmax over a question's slice is calibrated."""
    def __init__(self, name):
        super().__init__()
        kwargs = {"attn_implementation": "eager"} if "ettin" in name else {}
        self.encoder = AutoModel.from_pretrained(name, **kwargs)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.heads = nn.ModuleList([nn.Linear(hidden, len(QUESTIONS[q])) for q in QUESTION_IDS])
        self.register_buffer("temperature", torch.ones(TOTAL_OPTIONS))
    def forward(self, input_ids, attention_mask):
        states = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        mask = attention_mask.unsqueeze(-1).to(states.dtype)
        pooled = self.dropout((states * mask).sum(1) / mask.sum(1).clamp(min=1.0))
        return torch.cat([head(pooled) for head in self.heads], dim=-1) / self.temperature

def encode(tokenizer, texts):
    return tokenizer(texts, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")

def batches(rows, size, shuffle, seed=8):
    rows = list(rows)
    if shuffle:
        random.Random(seed).shuffle(rows)
    for start in range(0, len(rows), size):
        yield rows[start:start + size]

def probabilities(model, tokenizer, rows):
    """Per row: the softmax over its question's options."""
    model.eval()
    out = []
    with torch.no_grad():
        for chunk in batches(rows, 128, shuffle=False):
            ids = encode(tokenizer, [r["text"] for r in chunk]).to(DEVICE)
            logits = model(ids["input_ids"], ids["attention_mask"]).float().cpu()
            for row, line in zip(chunk, logits):
                start, end = OFFSETS[row["question"]]
                out.append(torch.softmax(line[start:end], -1).tolist())
    return out

def report(rows, probs):
    """Per question: per-class accuracy, confident coverage and confident accuracy."""
    result = {}
    for q in QUESTION_IDS:
        pairs = [(r, p) for r, p in zip(rows, probs) if r["question"] == q]
        if not pairs:
            continue
        options = QUESTIONS[q]
        classes, confident = {}, []
        for row, p in pairs:
            best = max(range(len(p)), key=p.__getitem__)
            hit = options[best] == row["label"]
            classes.setdefault(row["label"], []).append(hit)
            if p[best] >= MIN_CONFIDENCE:
                confident.append(hit)
        result[q] = {
            "classes": {k: sum(v) / len(v) for k, v in sorted(classes.items())},
            "coverage": len(confident) / len(pairs),
            "confidentAccuracy": sum(confident) / len(confident) if confident else 0.0,
            "rows": len(pairs),
        }
    return result

def worst_class(result):
    return min(min(r["classes"].values()) for r in result.values())

In [ ]:
def fit_temperatures(model, tokenizer, rows):
    """Per question, the temperature minimising validation negative log-likelihood (grid search)."""
    model.eval()
    model.temperature.fill_(1.0)
    with torch.no_grad():
        logits, labels = {q: [] for q in QUESTION_IDS}, {q: [] for q in QUESTION_IDS}
        for chunk in batches(rows, 128, shuffle=False):
            ids = encode(tokenizer, [r["text"] for r in chunk]).to(DEVICE)
            out = model(ids["input_ids"], ids["attention_mask"]).float().cpu()
            for row, line in zip(chunk, out):
                start, end = OFFSETS[row["question"]]
                logits[row["question"]].append(line[start:end])
                labels[row["question"]].append(QUESTIONS[row["question"]].index(row["label"]))
    temperatures = {}
    for q in QUESTION_IDS:
        if not logits[q]:
            temperatures[q] = 1.0
            continue
        x, y = torch.stack(logits[q]), torch.tensor(labels[q])
        grid = [0.5 + 0.1 * i for i in range(46)]
        temperatures[q] = min(grid, key=lambda t: nn.functional.cross_entropy(x / t, y).item())
        start, end = OFFSETS[q]
        model.temperature[start:end] = temperatures[q]
    return temperatures

def train_candidate(name):
    tokenizer = AutoTokenizer.from_pretrained(name, use_fast=True)
    model = DecisionModel(name).to(DEVICE)
    optimiser = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    steps = EPOCHS * math.ceil(len(SPLITS["train"]) / BATCH_SIZE)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimiser, max_lr=LEARNING_RATE, total_steps=steps, pct_start=0.1)
    best, best_state, stale, history = -1.0, None, 0, []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for chunk in batches(SPLITS["train"], BATCH_SIZE, shuffle=True, seed=epoch):
            ids = encode(tokenizer, [r["text"] for r in chunk]).to(DEVICE)
            logits = model(ids["input_ids"], ids["attention_mask"])
            loss = 0.0
            for i, row in enumerate(chunk):
                start, end = OFFSETS[row["question"]]
                target = torch.tensor([QUESTIONS[row["question"]].index(row["label"])], device=DEVICE)
                loss = loss + nn.functional.cross_entropy(logits[i:i + 1, start:end], target)
            (loss / len(chunk)).backward()
            optimiser.step(); scheduler.step(); optimiser.zero_grad()
        score = worst_class(report(SPLITS["validation"], probabilities(model, tokenizer, SPLITS["validation"])))
        history.append({"epoch": epoch, "worstClassValidation": score})
        print(f"  {name} epoch {epoch}: worst-class validation {score:.3f}")
        if score > best:
            best, stale = score, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
            if stale >= PATIENCE:
                break
    model.load_state_dict(best_state)
    temperatures = fit_temperatures(model, tokenizer, SPLITS["validation"])
    return model, tokenizer, {"history": history, "temperatures": temperatures, "bestWorstClassValidation": best}

In [ ]:
import onnx
import onnxruntime as ort
from onnxruntime.quantization import QuantType, quantize_dynamic

def export_onnx(model, tokenizer, out):
    out.mkdir(parents=True, exist_ok=True)
    model = model.float().cpu().eval()
    sample = encode(tokenizer, ["Decide whether to use Room or SQLDelight"])
    fp32 = out / "model-fp32.onnx"
    torch.onnx.export(
        model, (sample["input_ids"], sample["attention_mask"]), str(fp32),
        input_names=["input_ids", "attention_mask"], output_names=["logits"],
        dynamic_axes={"input_ids": {0: "batch", 1: "sequence"}, "attention_mask": {0: "batch", 1: "sequence"},
                      "logits": {0: "batch"}},
        opset_version=17, dynamo=False,
    )
    int8 = out / "model.onnx"
    quantize_dynamic(str(fp32), str(int8), weight_type=QuantType.QInt8, per_channel=True)
    tokenizer.save_pretrained(out)
    model.to(DEVICE)
    return fp32, int8

def onnx_probabilities(path, tokenizer, rows):
    session = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    out, started = [], time.time()
    for row in rows:  # one text per call, as on the phone
        ids = tokenizer([row["text"]], truncation=True, max_length=MAX_LENGTH, return_tensors="np")
        logits = session.run(["logits"], {"input_ids": ids["input_ids"].astype("int64"),
                                          "attention_mask": ids["attention_mask"].astype("int64")})[0][0]
        start, end = OFFSETS[row["question"]]
        z = logits[start:end] - logits[start:end].max()
        e = [math.exp(v) for v in z]
        out.append([v / sum(e) for v in e])
    return out, (time.time() - started) / max(1, len(rows)) * 1000

def accuracy(rows, probs):
    return sum(QUESTIONS[r["question"]][max(range(len(p)), key=p.__getitem__)] == r["label"] for r, p in zip(rows, probs)) / len(rows)

def gate(test, adversarial, float_accuracy, int8_accuracy):
    failures = []
    for q, r in test.items():
        for label, value in r["classes"].items():
            if value < GATES["min_class_accuracy"]:
                failures.append(f"{q} class {label}: {value:.3f} < {GATES['min_class_accuracy']}")
        if r["confidentAccuracy"] < GATES["min_confident_accuracy"]:
            failures.append(f"{q} confident accuracy {r['confidentAccuracy']:.3f} < {GATES['min_confident_accuracy']}")
    rows = [r for r in adversarial.values()]
    covered = sum(r["coverage"] * r["rows"] for r in rows)
    hits = sum(r["confidentAccuracy"] * r["coverage"] * r["rows"] for r in rows)
    adv = hits / covered if covered else 1.0
    if adv < GATES["min_adversarial_confident_accuracy"]:
        failures.append(f"adversarial confident accuracy {adv:.3f} < {GATES['min_adversarial_confident_accuracy']}")
    if float_accuracy - int8_accuracy > GATES["max_int8_accuracy_drop"]:
        failures.append(f"INT8 drops accuracy by {float_accuracy - int8_accuracy:.3f}")
    return failures, adv

In [ ]:
RESULTS_FILE = WORK / "results.json"
results = json.loads(RESULTS_FILE.read_text()) if RESULTS_FILE.exists() else {}
chosen = next((name for name in CANDIDATES if results.get(name, {}).get("passed")), None)
for name in CANDIDATES:
    if chosen and not TRAIN_ALL:
        break
    if name in results:
        print(f"{name}: already evaluated ({'PASS' if results[name]['passed'] else 'FAIL'})")
        continue
    print(f"== {name}")
    started = time.time()
    model, tokenizer, training = train_candidate(name)
    test = report(SPLITS["test"], probabilities(model, tokenizer, SPLITS["test"]))
    adversarial = report(SPLITS["adversarial"], probabilities(model, tokenizer, SPLITS["adversarial"]))
    out = WORK / "candidates" / name.replace("/", "__")
    fp32, int8 = export_onnx(model, tokenizer, out)
    float_probs = probabilities(model, tokenizer, SPLITS["test"])
    int8_probs, latency = onnx_probabilities(int8, tokenizer, SPLITS["test"])
    float_accuracy, int8_accuracy = accuracy(SPLITS["test"], float_probs), accuracy(SPLITS["test"], int8_probs)
    failures, adversarial_confident = gate(test, adversarial, float_accuracy, int8_accuracy)
    results[name] = {
        "passed": not failures, "failures": failures, "training": training,
        "test": test, "adversarial": adversarial, "adversarialConfidentAccuracy": adversarial_confident,
        "floatAccuracy": float_accuracy, "int8Accuracy": int8_accuracy,
        "int8Bytes": int8.stat().st_size, "cpuMillisPerText": latency, "minutes": (time.time() - started) / 60,
    }
    RESULTS_FILE.write_text(json.dumps(results, indent=2))
    print(f"{name}: {'PASS' if not failures else 'FAIL'}; INT8 {int8.stat().st_size / 1e6:.1f} MB, "
          f"{latency:.1f} ms/text on CPU, accuracy {float_accuracy:.3f} -> {int8_accuracy:.3f}")
    for failure in failures:
        print("   ", failure)
    if not failures and chosen is None:
        chosen = name
    del model; torch.cuda.empty_cache() if DEVICE == "cuda" else None

print("\n| candidate | passed | INT8 MB | CPU ms/text | test accuracy (INT8) | adversarial confident |")
print("|---|---|---|---|---|---|")
for name, r in results.items():
    print(f"| {name} | {'yes' if r['passed'] else 'no'} | {r['int8Bytes'] / 1e6:.1f} | {r['cpuMillisPerText']:.1f} | "
          f"{r['int8Accuracy']:.3f} | {r['adversarialConfidentAccuracy']:.3f} |")
print("chosen:", chosen or "none passed; nothing to release")

In [ ]:
def secret(name):
    if os.environ.get(name):
        return os.environ[name].strip()
    for loader in ("kaggle", "colab"):
        try:
            if loader == "kaggle":
                from kaggle_secrets import UserSecretsClient
                return UserSecretsClient().get_secret(name).strip()
            from google.colab import userdata
            return (userdata.get(name) or "").strip() or None
        except Exception:
            pass
    return None

def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()

ASSETS = WORK / "release-assets"
if chosen:
    shutil.rmtree(ASSETS, ignore_errors=True)
    package = ASSETS / "package"
    package.mkdir(parents=True)
    candidate = WORK / "candidates" / chosen.replace("/", "__")
    shutil.copy(candidate / "model.onnx", package / "model.onnx")
    shutil.copy(candidate / "tokenizer.json", package / "tokenizer.json")
    (package / "config.json").write_text(json.dumps({
        "schema": "aive-orchestration-decisions/1",
        "encoder": chosen,
        "maxLength": MAX_LENGTH,
        "minConfidence": MIN_CONFIDENCE,
        "questions": [{"id": q, "options": QUESTIONS[q], "offset": OFFSETS[q][0]} for q in QUESTION_IDS],
        "scores": {k: results[chosen][k] for k in ("test", "adversarial", "int8Accuracy", "cpuMillisPerText", "int8Bytes")},
    }, indent=2))
    asset = ASSETS / "aive-orchestration-decisions-int8.tar.gz"
    with tarfile.open(asset, "w:gz") as tar:
        for file in sorted(package.iterdir()):
            tar.add(file, arcname=file.name)
    catalog = {"releaseRepository": REPOSITORY, "releaseTag": RELEASE_TAG, "specialists": [{
        "specialistId": "orchestration:decisions",
        "mergedVariants": [{
            "logicalArtifactId": "orchestration:decisions:int8", "foundationModelId": chosen,
            "releaseRepository": REPOSITORY, "releaseTag": RELEASE_TAG, "assetName": asset.name,
            "sha256": sha256(asset), "format": "onnx", "precision": "int8", "kind": "MergedModel",
            "capabilities": ["orchestration-decisions"],
        }],
    }]}
    (ASSETS / "catalog.json").write_text(json.dumps(catalog, indent=2))
    shutil.copy(RESULTS_FILE, ASSETS / "results.json")
    print(f"{asset.name}: {asset.stat().st_size / 1e6:.1f} MB; register with "
          f"python3 tools/orchestration_training/register_catalog.py catalog.json")

In [ ]:
import urllib.request, urllib.error

def github(method, url, token, data=None, content_type="application/json"):
    request = urllib.request.Request(url, data=data, method=method, headers={
        "Authorization": f"Bearer {token}", "Accept": "application/vnd.github+json", "Content-Type": content_type})
    with urllib.request.urlopen(request, timeout=300) as response:
        body = response.read()
        return json.loads(body) if body else {}

token = secret("GITHUB_TOKEN")
if chosen and (UPLOAD is True or (UPLOAD == "auto" and token)):
    api = f"https://api.github.com/repos/{REPOSITORY}/releases"
    try:
        release = github("GET", f"{api}/tags/{RELEASE_TAG}", token)
    except urllib.error.HTTPError as failure:
        if failure.code != 404:
            raise
        release = github("POST", api, token, json.dumps({
            "tag_name": RELEASE_TAG, "name": RELEASE_TAG,
            "body": f"Aive orchestration decision model ({chosen}, INT8 ONNX). See catalog.json and results.json."}).encode())
    existing = {a["name"]: a for a in release.get("assets", [])}
    for path in (ASSETS / "aive-orchestration-decisions-int8.tar.gz", ASSETS / "catalog.json", ASSETS / "results.json"):
        if path.name in existing:
            github("DELETE", f"{api}/assets/{existing[path.name]['id']}", token)
        upload = release["upload_url"].split("{")[0] + f"?name={path.name}"
        github("POST", upload, token, path.read_bytes(), "application/octet-stream")
        print("uploaded", path.name)
elif chosen:
    print(f"Not uploaded (no GITHUB_TOKEN). Assets are in {ASSETS}")